In [ ]:
NOTEBOOK_VERSION = 'v13-apt-resilient-hevc-final'
print(f'CherryFlash {NOTEBOOK_VERSION}', flush=True)

import asyncio
import os
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from pathlib import Path

def run(cmd: list[str]) -> None:
    subprocess.run(cmd, check=True)

def disable_flaky_apt_repos() -> None:
    sources_dir = Path('/etc/apt/sources.list.d')
    if not sources_dir.exists():
        return
    for candidate in sources_dir.glob('*.list'):
        try:
            text = candidate.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            continue
        if 'cloud.r-project.org/bin/linux/ubuntu/jammy-cran40' not in text:
            continue
        disabled = candidate.with_suffix(candidate.suffix + '.disabled')
        if disabled.exists():
            disabled.unlink()
        candidate.rename(disabled)
        print(f'Disabled flaky apt repo: {candidate.name}', flush=True)

def apt_update_resilient() -> None:
    disable_flaky_apt_repos()
    last_error = None
    cmd = [
        'apt-get', 'update', '-qq',
        '-o', 'Acquire::Retries=3',
        '-o', 'Acquire::By-Hash=force',
    ]
    for attempt in range(1, 4):
        try:
            run(cmd)
            return
        except subprocess.CalledProcessError as exc:
            last_error = exc
            if attempt == 3:
                raise
            print(f'apt-get update failed (attempt {attempt}/3); retrying...', flush=True)
            time.sleep(5)
    if last_error is not None:
        raise last_error

apt_update_resilient()
run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libxrender1', 'libxi6', 'libxkbcommon-x11-0'])

blender_dir = Path('/opt/blender-4.5.4-linux-x64')
blender_bin = blender_dir / 'blender'
if not blender_bin.exists():
    run(['wget', '-q', 'https://download.blender.org/release/Blender4.5/blender-4.5.4-linux-x64.tar.xz', '-O', '/tmp/blender.tar.xz'])
    run(['tar', '-xf', '/tmp/blender.tar.xz', '-C', '/opt'])
    symlink_path = Path('/usr/local/bin/blender')
    if symlink_path.exists() or symlink_path.is_symlink():
        symlink_path.unlink()
    os.symlink(str(blender_bin), str(symlink_path))

run([sys.executable, '-m', 'pip', 'install', '-q', 'moviepy', 'pillow', 'numpy', 'imageio_ffmpeg', 'opencv-python', 'requests', 'telethon', 'cryptography'])
run(['blender', '--version'])


In [ ]:
import os
import json
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

EXPECTED_BUNDLE_ARCHIVE = 'cherryflash.zip'
EXPECTED_RENDER_SCRIPT = Path('scripts/render_cherryflash_full.py')

def _dataset_priority(name: str) -> int:
    lowered = name.strip().casefold()
    if lowered.startswith('cherryflash-session-'):
        return 0
    if lowered.startswith('cherryflash-runtime'):
        return 1
    if lowered.startswith('cherryflash'):
        return 2
    return 9

def _dataset_timestamp(name: str) -> int:
    match = re.search(r'(\\d+)$', name.strip())
    if not match:
        return 0
    try:
        return int(match.group(1))
    except Exception:
        return 0

def _bundle_sort_key(bundle_root: Path, *, input_root: Path) -> tuple[int, int, float]:
    try:
        top_name = bundle_root.relative_to(input_root).parts[0]
    except Exception:
        top_name = bundle_root.name
    try:
        mtime = bundle_root.stat().st_mtime
    except Exception:
        mtime = 0.0
    return (_dataset_priority(top_name), -_dataset_timestamp(top_name), -mtime)

def find_scripts_root(base: Path) -> Path | None:
    direct = base / EXPECTED_RENDER_SCRIPT
    if direct.exists():
        return base
    matches = sorted(base.rglob(str(EXPECTED_RENDER_SCRIPT)))
    if not matches:
        return None
    matches.sort(key=lambda path: (len(path.parts), path.as_posix()))
    return matches[0].parents[1]

def find_mounted_bundle(input_root: Path) -> Path | None:
    if not input_root.exists():
        return None
    candidates = []
    for top_level in sorted((p for p in input_root.iterdir() if p.is_dir()), key=lambda p: p.name):
        resolved = find_scripts_root(top_level)
        if resolved is not None:
            candidates.append(resolved)
    if not candidates:
        return None
    candidates.sort(key=lambda path: _bundle_sort_key(path, input_root=input_root))
    return candidates[0]

def pick_bundle_archive(input_root: Path, archive_name: str) -> Path | None:
    if not input_root.exists():
        return None
    archives = []
    for top_level in sorted((p for p in input_root.iterdir() if p.is_dir()), key=lambda p: p.name):
        for archive in sorted(top_level.rglob(archive_name)):
            archives.append((_bundle_sort_key(top_level, input_root=input_root), archive))
    if not archives:
        return None
    archives.sort(key=lambda item: item[0])
    return archives[0][1]

NOTEBOOK_ROOT = Path.cwd().resolve()
SOURCE_FOLDER = None
input_dir_names = []

INPUT_ROOT = Path('/kaggle/input')
top_level_dirs = sorted([p for p in INPUT_ROOT.iterdir() if p.is_dir()]) if INPUT_ROOT.exists() else []
input_dir_names = [p.name for p in top_level_dirs]
print('Kaggle input dirs:', input_dir_names, flush=True)
resolved = find_mounted_bundle(INPUT_ROOT)
if resolved is not None:
    SOURCE_FOLDER = resolved
    print(f'Using mounted CherryFlash bundle at {SOURCE_FOLDER}', flush=True)
else:
    archive_path = pick_bundle_archive(INPUT_ROOT, EXPECTED_BUNDLE_ARCHIVE) if INPUT_ROOT.exists() else None
    if archive_path is not None:
        extracted_root = Path('/kaggle/working/cherryflash_bundle')
        if extracted_root.exists():
            shutil.rmtree(extracted_root)
        extracted_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(extracted_root)
        resolved = find_scripts_root(extracted_root)
        if resolved is not None:
            SOURCE_FOLDER = resolved
            print(f'Using extracted CherryFlash bundle from {archive_path} at {SOURCE_FOLDER}', flush=True)

if SOURCE_FOLDER is None:
    for candidate in [NOTEBOOK_ROOT, Path('/kaggle/working')]:
        resolved = find_scripts_root(candidate)
        if resolved is not None:
            SOURCE_FOLDER = resolved
            print(f'Using local CherryFlash fallback bundle at {SOURCE_FOLDER}', flush=True)
            break

if SOURCE_FOLDER is None:
    raise RuntimeError(
        f'CherryFlash source was not found locally or under /kaggle/input. '
        f'Expected archive {EXPECTED_BUNDLE_ARCHIVE}; visible inputs={input_dir_names}'
    )

print(f'Using SOURCE_FOLDER={SOURCE_FOLDER}', flush=True)
os.environ['CHERRYFLASH_ROOT'] = str(SOURCE_FOLDER)
os.environ['CHERRYFLASH_ARTIFACTS_ROOT'] = '/kaggle/working/artifacts'
os.environ['BLENDER_BIN'] = shutil.which('blender') or '/usr/local/bin/blender'

def _log(message: str) -> None:
    print(message, flush=True)

def _run_async(awaitable):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(awaitable)

    result = {}
    error = {}

    def _runner():
        try:
            result['value'] = asyncio.run(awaitable)
        except Exception as exc:
            error['exc'] = exc

    thread = threading.Thread(target=_runner)
    thread.start()
    thread.join()
    if 'exc' in error:
        raise error['exc']
    return result.get('value')

def _ensure_story_publish_helper(source_folder: Path) -> bool:
    helper_path = source_folder / 'kaggle_common' / 'story_publish.py'
    if not helper_path.exists():
        _log('[SKIP] common story_publish helper not found in CherryFlash bundle.')
        return False
    target = Path('/kaggle/working/story_publish.py')
    shutil.copy2(helper_path, target)
    if str(target.parent) not in sys.path:
        sys.path.insert(0, str(target.parent))
    return True

story_publish_ready = _ensure_story_publish_helper(SOURCE_FOLDER)
preflight_story_publish_from_kaggle = None
publish_story_from_kaggle = None
if story_publish_ready:
    from story_publish import preflight_story_publish_from_kaggle, publish_story_from_kaggle

selection_manifest_path = SOURCE_FOLDER / 'assets' / 'cherryflash_selection.json'
selection_manifest = json.loads(selection_manifest_path.read_text(encoding='utf-8')) if selection_manifest_path.exists() else {}
story_publish_requested = bool((selection_manifest or {}).get('story_publish_enabled'))
story_search_roots = [SOURCE_FOLDER, INPUT_ROOT]
story_preflight = None
if story_publish_requested and preflight_story_publish_from_kaggle is None:
    raise RuntimeError('CherryFlash story publish requested but shared story helper is unavailable in Kaggle bundle')
if preflight_story_publish_from_kaggle is not None:
    story_preflight = _run_async(
        preflight_story_publish_from_kaggle(
            search_roots=story_search_roots,
            output_dir=Path('/kaggle/working'),
            log=_log,
        )
    )
    if story_preflight:
        _log(f"Story preflight status: {'OK' if story_preflight.get('ok') else 'FAIL'}")
        if not story_preflight.get('ok'):
            raise RuntimeError('CherryFlash story publish preflight failed')
if story_publish_requested and not story_preflight:
    raise RuntimeError('CherryFlash story publish requested but story_publish.json was not mounted into Kaggle input')

script_path = SOURCE_FOLDER / 'scripts' / 'render_cherryflash_full.py'
subprocess.run([sys.executable, str(script_path), '--final'], check=True)

artifact_dir = Path('/kaggle/working/artifacts/codex/cherryflash_full_final')
final_mp4 = artifact_dir / 'cherryflash_full_final.mp4'
if not final_mp4.exists():
    raise RuntimeError(f'Final CherryFlash full mp4 not found: {final_mp4}')

story_report = None
if publish_story_from_kaggle is not None and (story_preflight is None or story_preflight.get('ok')):
    story_posters_dir = SOURCE_FOLDER / 'assets' / 'posters'
    story_posters = sorted(story_posters_dir.glob('*')) if story_posters_dir.exists() else None
    intro_candidate = artifact_dir / 'approval_cover.png'
    story_report = _run_async(
        publish_story_from_kaggle(
            final_video_path=final_mp4,
            intro_path=intro_candidate if intro_candidate.exists() else None,
            posters=story_posters,
            search_roots=story_search_roots,
            output_dir=Path('/kaggle/working'),
            log=_log,
        )
    )
    if story_report:
        _log(f"Story publish status: {'OK' if story_report.get('ok') else 'FAIL'}")
        if not story_report.get('ok'):
            raise RuntimeError('CherryFlash story publish failed')

export_mp4 = Path('/kaggle/working/cherryflash_full_final.mp4')
shutil.copy2(final_mp4, export_mp4)

for extra_name in ['approval_cover.png', 'manifest_final.md']:
    extra_path = artifact_dir / extra_name
    if extra_path.exists():
        shutil.copy2(extra_path, Path('/kaggle/working') / extra_name)

manifest = {
    'ok': True,
    'source_folder': str(SOURCE_FOLDER),
    'artifact_dir': str(artifact_dir),
    'export_mp4': str(export_mp4),
    'story_publish_enabled': bool(story_preflight or story_report),
    'story_publish_requested': story_publish_requested,
}
Path('/kaggle/working/cherryflash_output_manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(manifest, ensure_ascii=False, indent=2), flush=True)
